# Búsqueda de hiperparámetros - RandomizedSearchCV + GridSearchCV

Estrategia de dos etapas para Logistic Regression, Decision Tree y Random Forest:

1. **RandomizedSearchCV** con rangos amplios (distribuciones continuas/discretas), para explorar el espacio de forma gruesa con `n_iter` controlado.
2. **GridSearchCV** acotado alrededor del óptimo encontrado por el Randomized, para afinar.

---
## Setup

In [3]:
import time
import warnings
import pandas as pd
from pathlib import Path
from scipy.stats import randint, uniform
import sys
import os

sys.path.append(os.path.abspath('..'))
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, GridSearchCV
from src.models.pipeline import get_phase1_pipeline, get_phase2_pipeline


warnings.filterwarnings('ignore')

RANDOM_STATE = 42
cv_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)



---
## Carga de datos

In [4]:
processed_dir = Path("..") / "data" / "processed"
p1_dir = processed_dir / "phase1"
p2_dir = processed_dir / "phase2"

# Fase 1
X_train_1 = pd.read_csv(p1_dir / "X_train.csv")
y_train_1 = pd.read_csv(p1_dir / "y_train.csv").squeeze()

# Fase 2
X_train_2 = pd.read_csv(p2_dir / "X_train.csv")
y_train_2 = pd.read_csv(p2_dir / "y_train.csv").squeeze()

print(f"Fase 1 -> train: {X_train_1.shape}")
print(f"Fase 2 -> train: {X_train_2.shape}")


Fase 1 -> train: (103904, 6)
Fase 2 -> train: (103904, 21)


---
## Espacios de búsqueda - Etapa 1 (RandomizedSearchCV)

In [5]:
def get_param_distributions():
    param_dist_lr = {
        'clf__C': uniform(0.001, 100),
        'clf__penalty': ['l2'],
        'clf__solver': ['lbfgs', 'liblinear'],
    }

    param_dist_dt = {
        'clf__max_depth': randint(5, 35),
        'clf__min_samples_split': randint(20, 400),
        'clf__min_samples_leaf': randint(10, 150),
        'clf__criterion': ['gini', 'entropy'],
    }

    param_dist_rf = {
        'clf__n_estimators': randint(80, 350),
        'clf__max_depth': randint(8, 30),
        'clf__min_samples_split': randint(20, 250),
        'clf__min_samples_leaf': randint(5, 100),
        'clf__max_features': ['sqrt', 'log2'],
    }

    return {
        'Logistic Regression': param_dist_lr,
        'Decision Tree': param_dist_dt,
        'Random Forest': param_dist_rf,
    }


---
## Grids refinados - Etapa 2 (GridSearchCV)

Construyen un grid fino alrededor de los `best_params_` encontrados por el Randomized. Los deltas (`±3`, `±20`, etc.) son genéricos; si los óptimos de Fase 1 y Fase 2 quedan muy distintos entre sí, puede convenir ajustar estos pasos a mano por fase.

In [6]:
def build_refined_grid_lr(best_params):
    C = best_params['clf__C']
    return {
        'clf__C': sorted(set([C * 0.5, C * 0.75, C, C * 1.5, C * 2])),
        'clf__penalty': ['l2'],
        'clf__solver': [best_params['clf__solver']],
    }

def build_refined_grid_dt(best_params):
    depth = best_params['clf__max_depth']
    split = best_params['clf__min_samples_split']
    leaf = best_params['clf__min_samples_leaf']
    return {
        'clf__max_depth': sorted(set([max(2, depth - 3), depth, depth + 3])),
        'clf__min_samples_split': sorted(set([max(2, split - 20), split, split + 20])),
        'clf__min_samples_leaf': sorted(set([max(1, leaf - 10), leaf, leaf + 10])),
        'clf__criterion': [best_params['clf__criterion']],
    }

def build_refined_grid_rf(best_params):
    n_est = best_params['clf__n_estimators']
    depth = best_params['clf__max_depth']
    split = best_params['clf__min_samples_split']
    return {
        'clf__n_estimators': sorted(set([max(10, n_est - 50), n_est, n_est + 50])),
        'clf__max_depth': sorted(set([max(3, depth - 3), depth, depth + 3])),
        'clf__min_samples_split': sorted(set([max(2, split - 20), split, split + 20])),
        'clf__max_features': [best_params['clf__max_features']],
    }

refined_grid_builders = {
    'Logistic Regression': build_refined_grid_lr,
    'Decision Tree': build_refined_grid_dt,
    'Random Forest': build_refined_grid_rf,
}


---
## Búsqueda en dos etapas

Toma `X_train, y_train` y los pipelines de cada modelo, corre Randomized -> Grid refinado, y devuelve todo lo necesario para inspeccionar o reentrenar después.

In [7]:
def run_two_stage_search(X_train, y_train, pipelines_by_name, n_iter=50, label=""):
    param_distributions = get_param_distributions()
    results = {}

    for model_name, pipeline in pipelines_by_name.items():
        print(f"\n{'='*60}")
        print(f"{label} — {model_name}")
        print(f"{'='*60}")

        # Etapa 1: RandomizedSearchCV
        print(f"\nEtapa 1: RandomizedSearchCV ({model_name})")
        t0 = time.time()

        rs = RandomizedSearchCV(
            estimator=pipeline,
            param_distributions=param_distributions[model_name],
            n_iter=n_iter,
            cv=cv_strat,
            scoring='f1_weighted',
            n_jobs=-1,
            random_state=RANDOM_STATE,
            refit=True,
            verbose=0,
        )
        rs.fit(X_train, y_train)
        elapsed_rs = time.time() - t0

        print(f"Mejores parámetros (Randomized): {rs.best_params_}")
        print(f"Mejor F1 (CV): {rs.best_score_:.4f}")
        print(f"Tiempo: {elapsed_rs:.1f}s")

        # Etapa 2: GridSearchCV refinado
        print(f"\nEtapa 2: GridSearchCV refinado ({model_name})")
        t0 = time.time()

        refined_grid = refined_grid_builders[model_name](rs.best_params_)
        print(f"Grid refinado: {refined_grid}")

        gs = GridSearchCV(
            estimator=pipeline,
            param_grid=refined_grid,
            cv=cv_strat,
            scoring='f1_weighted',
            n_jobs=-1,
            refit=True,
            verbose=0,
        )
        gs.fit(X_train, y_train)
        elapsed_gs = time.time() - t0

        improvement = gs.best_score_ - rs.best_score_
        print(f"Mejores parámetros finales: {gs.best_params_}")
        print(f"Mejor F1 (CV): {gs.best_score_:.4f}  (vs Randomized: {rs.best_score_:.4f}, mejora: {improvement:+.4f})")
        print(f"Tiempo: {elapsed_gs:.1f}s")

        results[model_name] = {
            'random_search': rs,
            'grid_search': gs,
            'best_estimator': gs.best_estimator_,
            'best_params': gs.best_params_,
            'best_score_cv': gs.best_score_,
            'random_score_cv': rs.best_score_,
            'improvement': improvement,
            'time_random': elapsed_rs,
            'time_grid': elapsed_gs,
        }

    return results


---
## Pipelines por fase

In [8]:
pipelines_phase1 = {
    'Logistic Regression': get_phase1_pipeline('lr'),
    'Decision Tree': get_phase1_pipeline('dt'),
    'Random Forest': get_phase1_pipeline('rf'),
}

pipelines_phase2 = {
    'Logistic Regression': get_phase2_pipeline('lr'),
    'Decision Tree': get_phase2_pipeline('dt'),
    'Random Forest': get_phase2_pipeline('rf'),
}


---
## Búsqueda — Fase 1 (pre-flight)

In [9]:
results_phase1 = run_two_stage_search(
    X_train_1, y_train_1, pipelines_phase1, n_iter=50, label="Fase 1"
)



Fase 1 — Logistic Regression

Etapa 1: RandomizedSearchCV (Logistic Regression)


Mejores parámetros (Randomized): {'clf__C': 3.4398521115218395, 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}
Mejor F1 (CV): 0.7545
Tiempo: 11.0s

Etapa 2: GridSearchCV refinado (Logistic Regression)
Grid refinado: {'clf__C': [1.7199260557609197, 2.5798890836413797, 3.4398521115218395, 5.159778167282759, 6.879704223043679], 'clf__penalty': ['l2'], 'clf__solver': ['liblinear']}
Mejores parámetros finales: {'clf__C': 3.4398521115218395, 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}
Mejor F1 (CV): 0.7545  (vs Randomized: 0.7545, mejora: +0.0000)
Tiempo: 1.3s

Fase 1 — Decision Tree

Etapa 1: RandomizedSearchCV (Decision Tree)
Mejores parámetros (Randomized): {'clf__criterion': 'entropy', 'clf__max_depth': 17, 'clf__min_samples_leaf': 91, 'clf__min_samples_split': 386}
Mejor F1 (CV): 0.7844
Tiempo: 6.3s

Etapa 2: GridSearchCV refinado (Decision Tree)
Grid refinado: {'clf__max_depth': [14, 17, 20], 'clf__min_samples_split': [366, 386, 406], 'clf__min_samples_leaf': [81, 91, 101], 'c

---
## Búsqueda — Fase 2 (pre+post-flight)

In [10]:
results_phase2 = run_two_stage_search(
    X_train_2, y_train_2, pipelines_phase2, n_iter=50, label="Fase 2"
)



Fase 2 — Logistic Regression

Etapa 1: RandomizedSearchCV (Logistic Regression)


Mejores parámetros (Randomized): {'clf__C': 0.5532117123602399, 'clf__penalty': 'l2', 'clf__solver': 'lbfgs'}
Mejor F1 (CV): 0.8356
Tiempo: 40.3s

Etapa 2: GridSearchCV refinado (Logistic Regression)
Grid refinado: {'clf__C': [0.27660585618011996, 0.41490878427017996, 0.5532117123602399, 0.8298175685403599, 1.1064234247204798], 'clf__penalty': ['l2'], 'clf__solver': ['lbfgs']}
Mejores parámetros finales: {'clf__C': 0.41490878427017996, 'clf__penalty': 'l2', 'clf__solver': 'lbfgs'}
Mejor F1 (CV): 0.8357  (vs Randomized: 0.8356, mejora: +0.0000)
Tiempo: 3.3s

Fase 2 — Decision Tree

Etapa 1: RandomizedSearchCV (Decision Tree)
Mejores parámetros (Randomized): {'clf__criterion': 'gini', 'clf__max_depth': 32, 'clf__min_samples_leaf': 17, 'clf__min_samples_split': 54}
Mejor F1 (CV): 0.9408
Tiempo: 21.0s

Etapa 2: GridSearchCV refinado (Decision Tree)
Grid refinado: {'clf__max_depth': [29, 32, 35], 'clf__min_samples_split': [34, 54, 74], 'clf__min_samples_leaf': [7, 17, 27], 'clf__criterion':

---
## Resumen final por fase

In [11]:
def summarize_results(results, label=""):
    rows = []
    for model_name, r in results.items():
        rows.append({
            'Modelo': model_name,
            'F1 Randomized (CV)': r['random_score_cv'],
            'F1 Grid refinado (CV)': r['best_score_cv'],
            'Mejora': r['improvement'],
            'Tiempo Randomized (s)': r['time_random'],
            'Tiempo Grid (s)': r['time_grid'],
        })
    df = pd.DataFrame(rows).set_index('Modelo')
    print(f"\nResumen - {label}")
    return df.round(4)

summary_phase1 = summarize_results(results_phase1, "Fase 1")
summary_phase1



Resumen - Fase 1


,F1 Randomized (CV),F1 Grid refinado (CV),Mejora,Tiempo Randomized (s),Tiempo Grid (s)
Modelo,,,,,
Logistic Regression,0.7545,0.7545,0.0000,11.0045,1.2597
Decision Tree,0.7844,0.7846,0.0002,6.3175,3.3797
Random Forest,0.7850,0.7854,0.0003,304.7726,144.1821


In [12]:
summary_phase2 = summarize_results(results_phase2, "Fase 2")
summary_phase2



Resumen - Fase 2


,F1 Randomized (CV),F1 Grid refinado (CV),Mejora,Tiempo Randomized (s),Tiempo Grid (s)
Modelo,,,,,
Logistic Regression,0.8356,0.8357,0.000,40.3472,3.2851
Decision Tree,0.9408,0.9427,0.002,20.9643,12.3886
Random Forest,0.9404,0.9454,0.005,428.1138,330.3930


---
## Guardado de los mejores parámetros

Para usar después en `train.py` o en un notebook de evaluación en test.

In [13]:
import json

best_params_summary = {
    'phase1': {name: r['best_params'] for name, r in results_phase1.items()},
    'phase2': {name: r['best_params'] for name, r in results_phase2.items()},
}

output_dir = Path("models")
output_dir.mkdir(exist_ok=True)

with open(output_dir / "best_hyperparameters.json", "w", encoding="utf-8") as f:
    json.dump(best_params_summary, f, indent=4)

print("Hiperparámetros guardados en models/best_hyperparameters.json")
print(json.dumps(best_params_summary, indent=2))


Hiperparámetros guardados en models/best_hyperparameters.json
{
  "phase1": {
    "Logistic Regression": {
      "clf__C": 3.4398521115218395,
      "clf__penalty": "l2",
      "clf__solver": "liblinear"
    },
    "Decision Tree": {
      "clf__criterion": "entropy",
      "clf__max_depth": 14,
      "clf__min_samples_leaf": 101,
      "clf__min_samples_split": 406
    },
    "Random Forest": {
      "clf__max_depth": 13,
      "clf__max_features": "log2",
      "clf__min_samples_split": 143,
      "clf__n_estimators": 126
    }
  },
  "phase2": {
    "Logistic Regression": {
      "clf__C": 0.41490878427017996,
      "clf__penalty": "l2",
      "clf__solver": "lbfgs"
    },
    "Decision Tree": {
      "clf__criterion": "gini",
      "clf__max_depth": 32,
      "clf__min_samples_leaf": 7,
      "clf__min_samples_split": 34
    },
    "Random Forest": {
      "clf__max_depth": 24,
      "clf__max_features": "sqrt",
      "clf__min_samples_split": 102,
      "clf__n_estimators": 227
  